In [2]:
# Imports e criação do schema

import duckdb
import os

DB_PATH = "../fundos_cvm.duckdb"

try:
    con.close()
except NameError:
    pass

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

con = duckdb.connect(DB_PATH)

with open("../sql/schema.sql", "r", encoding="utf-8") as f:
    schema_sql = f.read()

con.execute(schema_sql)

print("Tabelas criadas:")
print(con.execute("SHOW TABLES").fetchdf())

Tabelas criadas:
                name
0      administrador
1             classe
2   classe_prestador
3              fundo
4             gestor
5     informe_diario
6  prestador_servico


In [3]:
# Carga de administrador

con.execute("""
    INSERT INTO administrador
    SELECT DISTINCT
        "CNPJ_Administrador" AS cnpj_administrador,
        "Administrador"      AS nome
    FROM read_csv_auto('../data/raw/registro_fundo.csv',
                        delim=';', header=True, encoding='latin-1')
    WHERE "CNPJ_Administrador" IS NOT NULL
""")

print(f"Administradores carregados: {con.execute('SELECT COUNT(*) FROM administrador').fetchone()[0]}")

Administradores carregados: 255


In [4]:
# Carga de gestor

con.execute("""
    INSERT INTO gestor
    SELECT DISTINCT
        "CPF_CNPJ_Gestor"     AS cpf_cnpj_gestor,
        "Gestor"              AS nome,
        "Tipo_Pessoa_Gestor"  AS tipo_pessoa
    FROM read_csv_auto('../data/raw/registro_fundo.csv',
                        delim=';', header=True, encoding='latin-1')
    WHERE "CPF_CNPJ_Gestor" IS NOT NULL
""")

print(f"Gestores carregados: {con.execute('SELECT COUNT(*) FROM gestor').fetchone()[0]}")

Gestores carregados: 1742


In [5]:
# Carga de prestador_servico

con.execute("""
    INSERT INTO prestador_servico
    SELECT DISTINCT cnpj, nome FROM (
        SELECT "CNPJ_Auditor" AS cnpj, "Auditor" AS nome
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Auditor" IS NOT NULL

        UNION

        SELECT "CNPJ_Custodiante" AS cnpj, "Custodiante" AS nome
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Custodiante" IS NOT NULL

        UNION

        SELECT "CNPJ_Controlador" AS cnpj, "Controlador" AS nome
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Controlador" IS NOT NULL
    )
""")

print(f"Prestadores carregados: {con.execute('SELECT COUNT(*) FROM prestador_servico').fetchone()[0]}")

Prestadores carregados: 158


In [6]:
#violações de data

total = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto(
        '../data/raw/registro_fundo.csv',
        delim=';', header=True, encoding='latin-1')
""").fetchone()[0]

violacoes = con.execute("""
    SELECT COUNT(*)
    FROM read_csv_auto('../data/raw/registro_fundo.csv',
                        delim=';', header=True, encoding='latin-1')
    WHERE "Data_Constituicao" > "Data_Registro"
""").fetchone()[0]

pct = (violacoes / total) * 100
print(f"Total de fundos: {total}")
print(f"'Violações' Data_Constituicao > Data_Registro: {violacoes} ({pct:.2f}%)")
print()
print(con.execute("""
    SELECT "Situacao", COUNT(*) AS qtd
    FROM read_csv_auto('../data/raw/registro_fundo.csv',
                        delim=';', header=True, encoding='latin-1')
    WHERE "Data_Constituicao" > "Data_Registro"
    GROUP BY "Situacao"
    ORDER BY qtd DESC
""").fetchdf())

Total de fundos: 88749
'Violações' Data_Constituicao > Data_Registro: 1642 (1.85%)

                  Situacao   qtd
0                Cancelado  1158
1  Em Funcionamento Normal   410
2            Em Liquidação    69
3             Incorporação     4
4     Em Situação Especial     1


In [7]:
# Carga de fundo

con.execute("""
    INSERT INTO fundo
    SELECT
        id_registro_fundo, cnpj_fundo, denominacao_social, tipo_fundo,
        situacao, data_registro, data_constituicao,
        cnpj_administrador, cpf_cnpj_gestor
    FROM (
        SELECT
            "ID_Registro_Fundo"   AS id_registro_fundo,
            "CNPJ_Fundo"          AS cnpj_fundo,
            "Denominacao_Social"  AS denominacao_social,
            "Tipo_Fundo"          AS tipo_fundo,
            "Situacao"            AS situacao,
            "Data_Registro"       AS data_registro,
            "Data_Constituicao"   AS data_constituicao,
            "CNPJ_Administrador"  AS cnpj_administrador,
            "CPF_CNPJ_Gestor"     AS cpf_cnpj_gestor,
            ROW_NUMBER() OVER (
                PARTITION BY "ID_Registro_Fundo"
                ORDER BY "CPF_CNPJ_Gestor"
            ) AS rn
        FROM read_csv_auto('../data/raw/registro_fundo.csv',
                            delim=';', header=True, encoding='latin-1')
    )
    WHERE rn = 1
""")

print(f"Fundos carregados: {con.execute('SELECT COUNT(*) FROM fundo').fetchone()[0]}")

Fundos carregados: 87636


In [8]:
#Carga de classe

con.execute("""
    INSERT INTO classe
    SELECT
        id_registro_classe, id_registro_fundo, cnpj_classe,
        denominacao_social, tipo_classe, situacao, classificacao,
        classificacao_anbima, publico_alvo, patrimonio_liquido
    FROM (
        SELECT
            "ID_Registro_Classe"  AS id_registro_classe,
            "ID_Registro_Fundo"   AS id_registro_fundo,
            printf('%02d.%03d.%03d/%04d-%02d',
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 1, 2) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 3, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 6, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 9, 4) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 13, 2) AS INTEGER)
            ) AS cnpj_classe,
            "Denominacao_Social"     AS denominacao_social,
            "Tipo_Classe"            AS tipo_classe,
            "Situacao"               AS situacao,
            "Classificacao"         AS classificacao,
            "Classificacao_Anbima"  AS classificacao_anbima,
            "Publico_Alvo"          AS publico_alvo,
            "Patrimonio_Liquido"    AS patrimonio_liquido,
            ROW_NUMBER() OVER (
                PARTITION BY printf('%02d.%03d.%03d/%04d-%02d',
                    CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 1, 2) AS INTEGER),
                    CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 3, 3) AS INTEGER),
                    CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 6, 3) AS INTEGER),
                    CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 9, 4) AS INTEGER),
                    CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 13, 2) AS INTEGER)
                )
                ORDER BY "ID_Registro_Classe"
            ) AS rn
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
    )
    WHERE rn = 1
""")

print(f"Classes carregadas: {con.execute('SELECT COUNT(*) FROM classe').fetchone()[0]}")

Classes carregadas: 36226


In [9]:
#Carga de classe_prestador

con.execute("""
    INSERT INTO classe_prestador
    SELECT DISTINCT cnpj_classe, cnpj_prestador, tipo
    FROM (
        SELECT
            printf('%02d.%03d.%03d/%04d-%02d',
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 1, 2) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 3, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 6, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 9, 4) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 13, 2) AS INTEGER)
            ) AS cnpj_classe,
            "CNPJ_Auditor" AS cnpj_prestador,
            'AUDITOR'      AS tipo
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Auditor" IS NOT NULL

        UNION ALL

        SELECT
            printf('%02d.%03d.%03d/%04d-%02d',
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 1, 2) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 3, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 6, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 9, 4) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 13, 2) AS INTEGER)
            ) AS cnpj_classe,
            "CNPJ_Custodiante" AS cnpj_prestador,
            'CUSTODIANTE'      AS tipo
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Custodiante" IS NOT NULL

        UNION ALL

        SELECT
            printf('%02d.%03d.%03d/%04d-%02d',
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 1, 2) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 3, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 6, 3) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 9, 4) AS INTEGER),
                CAST(SUBSTRING(LPAD(CAST("CNPJ_Classe" AS VARCHAR), 14, '0'), 13, 2) AS INTEGER)
            ) AS cnpj_classe,
            "CNPJ_Controlador" AS cnpj_prestador,
            'CONTROLADOR'      AS tipo
        FROM read_csv_auto('../data/raw/registro_classe.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "CNPJ_Controlador" IS NOT NULL
    )
""")

print(f"Vínculos classe-prestador carregados: {con.execute('SELECT COUNT(*) FROM classe_prestador').fetchone()[0]}")

Vínculos classe-prestador carregados: 98542


In [10]:
con.execute("""
    INSERT INTO informe_diario
    SELECT
        cnpj_classe, id_subclasse, dt_comptc, vl_total, vl_quota,
        vl_patrim_liq, captc_dia, resg_dia, nr_cotst
    FROM (
        SELECT
            "CNPJ_FUNDO_CLASSE"            AS cnpj_classe,
            COALESCE("ID_SUBCLASSE", '')   AS id_subclasse,
            "DT_COMPTC"                    AS dt_comptc,
            "VL_TOTAL"                     AS vl_total,
            "VL_QUOTA"                     AS vl_quota,
            "VL_PATRIM_LIQ"                AS vl_patrim_liq,
            "CAPTC_DIA"                    AS captc_dia,
            "RESG_DIA"                     AS resg_dia,
            "NR_COTST"                     AS nr_cotst,
            ROW_NUMBER() OVER (
                PARTITION BY "CNPJ_FUNDO_CLASSE", COALESCE("ID_SUBCLASSE", ''), "DT_COMPTC"
                ORDER BY "VL_QUOTA"
            ) AS rn
        FROM read_csv_auto('../data/raw/inf_diario_fi_2025*.csv',
                            delim=';', header=True, encoding='latin-1')
        WHERE "TP_FUNDO_CLASSE" = 'CLASSES - FIF'
          AND "CNPJ_FUNDO_CLASSE" IN (SELECT cnpj_classe FROM classe)
          AND "VL_QUOTA" < 1000000000
    )
    WHERE rn = 1
""")

print(f"Informes carregados: {con.execute('SELECT COUNT(*) FROM informe_diario').fetchone()[0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Informes carregados: 4500625


In [11]:
# Resumo final

print(con.execute("SHOW TABLES").fetchdf())
print()
for tabela in ['administrador', 'gestor', 'prestador_servico', 'fundo',
               'classe', 'classe_prestador', 'informe_diario']:
    qtd = con.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {qtd:,} registros")

                name
0      administrador
1             classe
2   classe_prestador
3              fundo
4             gestor
5     informe_diario
6  prestador_servico

administrador: 255 registros
gestor: 1,742 registros
prestador_servico: 158 registros
fundo: 87,636 registros
classe: 36,226 registros
classe_prestador: 98,542 registros
informe_diario: 4,500,625 registros


In [12]:
total_arquivo = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/inf_diario_fi_2025*.csv',
                                        delim=';', header=True, encoding='latin-1')
""").fetchone()[0]

tipo_fif = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/inf_diario_fi_2025*.csv',
                                        delim=';', header=True, encoding='latin-1')
    WHERE "TP_FUNDO_CLASSE" = 'CLASSES - FIF'
""").fetchone()[0]

sem_cadastro = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/inf_diario_fi_2025*.csv',
                                        delim=';', header=True, encoding='latin-1')
    WHERE "TP_FUNDO_CLASSE" = 'CLASSES - FIF'
      AND "CNPJ_FUNDO_CLASSE" NOT IN (SELECT cnpj_classe FROM classe)
""").fetchone()[0]

outliers = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/inf_diario_fi_2025*.csv',
                                        delim=';', header=True, encoding='latin-1')
    WHERE "TP_FUNDO_CLASSE" = 'CLASSES - FIF'
      AND "CNPJ_FUNDO_CLASSE" IN (SELECT cnpj_classe FROM classe)
      AND "VL_QUOTA" >= 1000000000
""").fetchone()[0]

carregados = con.execute("SELECT COUNT(*) FROM informe_diario").fetchone()[0]
duplicatas_conflitantes = tipo_fif - sem_cadastro - outliers - carregados

print(f"Total no arquivo (FI + CLASSES-FIF + FAPI):  {total_arquivo:,}")
print(f"Tipo CLASSES - FIF:                          {tipo_fif:,}")
print(f"  Descartados - sem cadastro de classe:      {sem_cadastro:,}")
print(f"  Descartados - outlier VL_QUOTA >= 1 bi:    {outliers:,}")
print(f"  Descartados - duplicatas conflitantes:     {duplicatas_conflitantes:,}")
print(f"Carregados:                                  {carregados:,}")

Total no arquivo (FI + CLASSES-FIF + FAPI):  6,385,728
Tipo CLASSES - FIF:                          4,884,419
  Descartados - sem cadastro de classe:      383,778
  Descartados - outlier VL_QUOTA >= 1 bi:    5
  Descartados - duplicatas conflitantes:     11
Carregados:                                  4,500,625


In [13]:
fundo_total = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/registro_fundo.csv',
                                        delim=';', header=True, encoding='latin-1')
""").fetchone()[0]
fundo_carregados = con.execute("SELECT COUNT(*) FROM fundo").fetchone()[0]

classe_total = con.execute("""
    SELECT COUNT(*) FROM read_csv_auto('../data/raw/registro_classe.csv',
                                        delim=';', header=True, encoding='latin-1')
""").fetchone()[0]
classe_carregados = con.execute("SELECT COUNT(*) FROM classe").fetchone()[0]

print(f"Fundo:  {fundo_total:,} lidos | {fundo_total - fundo_carregados:,} descartados (duplicata de gestor) | {fundo_carregados:,} carregados")
print(f"Classe: {classe_total:,} lidos | {classe_total - classe_carregados:,} descartados (duplicata de CNPJ) | {classe_carregados:,} carregados")

Fundo:  88,749 lidos | 1,113 descartados (duplicata de gestor) | 87,636 carregados
Classe: 36,239 lidos | 13 descartados (duplicata de CNPJ) | 36,226 carregados


In [14]:
con.close()